In [5]:
# =============================================================
# EDA — Dataset 3: Red Hat Jira (ZIP/CSV) — ShiftMetrics Analytics
# Dataset: 2 archivos ZIP, 266 MB
# Bucket:  gs://shiftmetrics-bronze/redhat-jira/
# Objetivo: tipos de issues, fechas, estados, nulos, cycle time
# =============================================================

*Auth + Imports combinados en celda siguiente*

In [3]:
# ── 0. Autenticación GCP + Imports ───────────────────────────────────────────
import os

GCP_PROJECT = 'shiftmetrics-analytics'
GCS_BUCKET = 'shiftmetrics-bronze'
GCS_PREFIX = 'redhat-jira/'

credentials = None
active_project = None

try:
    from google.colab import auth  # type: ignore
    auth.authenticate_user()
    print('✅ Autenticado en Colab')
    try:
        import google.auth
        from google.auth.transport.requests import Request
        credentials, active_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
        if hasattr(credentials, 'refresh'):
            credentials.refresh(Request())
        print(f'✅ Credenciales ADC listas (proyecto detectado: {active_project})')
    except Exception as exc:
        print(f'⚠️ Colab autenticado, pero no se pudieron cargar ADC: {exc}')
except ImportError:
    try:
        import google.auth
        from google.auth.transport.requests import Request
        credentials, active_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
        if hasattr(credentials, 'refresh'):
            credentials.refresh(Request())
        print('✅ Autenticación ADC detectada localmente')
        print(f'✅ Proyecto ADC: {active_project}')
    except Exception as exc:
        print('❌ No se encontraron credenciales de Google Cloud.')
        print('   Ejecuta una de estas opciones en la terminal antes de correr el notebook:')
        print('   - gcloud auth application-default login')
        print('   - export GOOGLE_APPLICATION_CREDENTIALS="/ruta/a/service-account.json"')
        raise RuntimeError(f'Credenciales GCP ausentes o inválidas: {exc}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage
import io, os, zipfile, tempfile

pd.set_option('display.max_columns', 50)
sns.set_theme(style='darkgrid')

PROJECT = GCP_PROJECT
BUCKET = GCS_BUCKET
PREFIX = GCS_PREFIX

print('✅ Imports OK')
print(f'✅ Proyecto de trabajo: {PROJECT}')

✅ Autenticación ADC detectada localmente
✅ Proyecto ADC: None
✅ Imports OK
✅ Proyecto de trabajo: shiftmetrics-analytics


In [4]:
# ── 1. ejecución / configuracion ───────────────────────────────────────────
# Modo de ejecución: 'colab' => muestra segura para EDA; 'local' => procesa por chunks y vuelca parquet
MODE = 'local'   # cambiar a 'local' si ejecutas en una máquina con suficiente RAM o disco
CHUNK_SIZE = 200_000
SAMPLE_HEAD = 500
SAMPLE_FRAC = 0.02
LOCAL_OUTPUT_DIR = 'redhat_parquet'

import os
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

print(f"MODE={MODE}  CHUNK_SIZE={CHUNK_SIZE}  SAMPLE_HEAD={SAMPLE_HEAD}  SAMPLE_FRAC={SAMPLE_FRAC}")

# Nota: en Colab deja 'colab' (genera muestras pequeñas). En local pon 'local' para crear parquet por chunks.


MODE=local  CHUNK_SIZE=200000  SAMPLE_HEAD=500  SAMPLE_FRAC=0.02


In [5]:
# ── 2. Listar archivos en Bronze/redhat-jira ──────────────────────────────────
client = storage.Client(project=PROJECT, credentials=credentials)
bucket = client.bucket(BUCKET)

blobs = list(bucket.list_blobs(prefix=PREFIX))
print(f"Archivos en redhat-jira/: {len(blobs)}")
for b in blobs:
    print(f"  {b.name:50s}  {b.size/(1024**2):.1f} MB  [{b.content_type}]")
zip_blobs = [b for b in blobs if b.name.endswith('.zip')]
print(f'ZIPs encontrados: {len(zip_blobs)}')


Archivos en redhat-jira/: 2
  redhat-jira/redhat-inputs.zip                       5.7 MB  [application/zip]
  redhat-jira/redhat-outputs.zip                      248.2 MB  [application/zip]
ZIPs encontrados: 2


In [6]:
# ── Memory check (run before heavy I/O) ───────────────────────────────────
try:
    import psutil
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psutil", "--quiet"]) 
    import psutil

mem = psutil.virtual_memory()
print(f"RAM total: {mem.total/(1024**3):.1f} GB — disponible: {mem.available/(1024**3):.1f} GB — uso: {mem.percent}%")

# Helper: small sampler for chunks → returns small DataFrame representative of file
from io import TextIOWrapper

def sample_from_zf(zf, csv_name, enc_candidates=['utf-8','latin-1'], nrows_head=500, chunk_sample_frac=0.02, chunksize=200_000):
    """Try to read a small sample (nrows_head) using encodings; if file is large, iterate chunks and keep small fraction sample."""
    for enc in enc_candidates:
        try:
            with zf.open(csv_name) as byte_f:
                text_f = TextIOWrapper(byte_f, encoding=enc)
                # Try quick head for schema
                try:
                    df_head = pd.read_csv(text_f, nrows=nrows_head, on_bad_lines='skip')
                except Exception:
                    # fallback: try reading small chunk via chunksize
                    byte_f.seek(0)
                    text_f = TextIOWrapper(byte_f, encoding=enc)
                    reader = pd.read_csv(text_f, chunksize=chunksize, on_bad_lines='skip')
                    df_head = next(reader)
                return df_head
        except Exception:
            continue
    raise ValueError(f"Could not read {csv_name} with provided encodings")


RAM total: 30.9 GB — disponible: 11.5 GB — uso: 62.8%


In [7]:
# ── 3. Descomprimir y cargar AMBOS ZIPs (CHUNKED, memory-safe, modo-aware) ───
import tempfile
import zipfile
import os
import gc
from io import TextIOWrapper

# Asegurar pyarrow si vamos a escribir parquet en modo local
if MODE == 'local':
    try:
        import pyarrow  # noqa: F401
    except Exception:
        import sys, subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow', '--quiet'])


dfs = []
zip_summaries = []

# Asegurar que `zip_blobs` existe (la sesión de Colab puede haber perdido variables)
if 'zip_blobs' not in locals():
    try:
        zip_blobs = [b for b in blobs if b.name.endswith('.zip')]
        print(f"Construido zip_blobs a partir de `blobs`: {len(zip_blobs)} ZIP(s) encontrados")
    except NameError:
        print("Variable 'blobs' no encontrada: ejecuta primero la celda que lista archivos (listado en GCS).")
        zip_blobs = []

if not zip_blobs:
    print("No hay archivos ZIP para procesar. Revisa las variables o ejecuta la celda de listado de blobs.")
else:
    for zip_blob in zip_blobs:
        zip_name = os.path.basename(zip_blob.name)
        size_mb = getattr(zip_blob, 'size', 0) / (1024**2)
        print(f"\n{'='*60}")
        print(f"Procesando: {zip_name} ({size_mb:.1f} MB)")
        print(f"{'='*60}")

        with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp:
            zip_blob.download_to_filename(tmp.name)
            with zipfile.ZipFile(tmp.name) as zf:
                csv_names = [n for n in zf.namelist() if n.endswith('.csv')]
                print(f"  Archivos en ZIP: {len(zf.namelist())} | CSVs: {len(csv_names)}")

                loaded = 0
                for csv_name in csv_names:
                    try:
                        if MODE == 'colab':
                            # Lectura segura para EDA: solo head o small sample
                            try:
                                sample_df = sample_from_zf(zf, csv_name, enc_candidates=['utf-8','latin-1'], nrows_head=SAMPLE_HEAD)
                            except Exception as e:
                                print(f"  WARN: no se pudo leer head de {csv_name}: {e}. Intentando primer chunk...")
                                # Fallback: leer primer chunk
                                with zf.open(csv_name) as byte_f:
                                    for enc in ['utf-8','latin-1']:
                                        try:
                                            text_f = TextIOWrapper(byte_f, encoding=enc)
                                            reader = pd.read_csv(text_f, chunksize=CHUNK_SIZE, on_bad_lines='skip')
                                            first = next(reader)
                                            sample_df = first.head(SAMPLE_HEAD)
                                            break
                                        except Exception:
                                            byte_f.seek(0)
                                            continue

                            sample_df['_source_file'] = os.path.basename(csv_name)
                            sample_df['_zip_source'] = zip_name

                            if len(sample_df) > SAMPLE_HEAD:
                                sample_df = sample_df.sample(frac=SAMPLE_FRAC, random_state=42)

                            dfs.append(sample_df)
                            print(f"  OK {os.path.basename(csv_name)}: muestra {sample_df.shape[0]:,}r x {sample_df.shape[1]}c")
                            loaded += 1

                        else:  # MODE == 'local' -> procesar por chunks y volcar a parquet por chunks
                            chunk_i = 0
                            with zf.open(csv_name) as byte_f:
                                for enc in ['utf-8','latin-1']:
                                    try:
                                        text_f = TextIOWrapper(byte_f, encoding=enc)
                                        reader = pd.read_csv(text_f, chunksize=CHUNK_SIZE, on_bad_lines='skip')
                                        for chunk in reader:
                                            out_name = f"{LOCAL_OUTPUT_DIR}/{zip_name.replace('.zip','')}_{os.path.basename(csv_name).replace('.csv','')}_chunk{chunk_i}.parquet"
                                            chunk.to_parquet(out_name, index=False)
                                            print(f"   WROTE: {out_name} ({len(chunk):,} rows)")
                                            chunk_i += 1
                                        break
                                    except Exception:
                                        byte_f.seek(0)
                                        continue
                            loaded += 1

                    except Exception as e:
                        print(f"  SKIP {csv_name}: {e}")

                zip_summaries.append({
                    'zip': zip_name, 'size_mb': size_mb,
                    'csv_files': len(csv_names), 'loaded': loaded
                })
            os.unlink(tmp.name)

    # Consolidar muestras en un solo DataFrame representativo para el EDA (solo en colab)
    if MODE == 'colab':
        if dfs:
            df_all = pd.concat(dfs, ignore_index=True)
            del dfs
            gc.collect()
            print(f"\nTotal filas (muestra combinada) Red Hat: {len(df_all):,}")
            dfs = [df_all]
        else:
            print("No se cargaron DataFrames en modo 'colab'")
    else:
        print(f"Modo 'local': archivos parquet escritos en: {LOCAL_OUTPUT_DIR}")



Procesando: redhat-inputs.zip (5.7 MB)
  Archivos en ZIP: 257 | CSVs: 251
   WROTE: redhat_parquet/redhat-inputs_WFCC_chunk0.parquet (302 rows)
   WROTE: redhat_parquet/redhat-inputs_WFCOM_chunk0.parquet (69 rows)
   WROTE: redhat_parquet/redhat-inputs_WFCORE_chunk0.parquet (6,002 rows)
   WROTE: redhat_parquet/redhat-inputs_WFDISC_chunk0.parquet (50 rows)
   WROTE: redhat_parquet/redhat-inputs_WFMP_chunk0.parquet (270 rows)
   WROTE: redhat_parquet/redhat-inputs_WFNC_chunk0.parquet (88 rows)
   WROTE: redhat_parquet/redhat-inputs_WFSSL_chunk0.parquet (110 rows)
   WROTE: redhat_parquet/redhat-inputs_WFTC_chunk0.parquet (126 rows)
   WROTE: redhat_parquet/redhat-inputs_WFWIP_chunk0.parquet (479 rows)
   WROTE: redhat_parquet/redhat-inputs_WINDUP_chunk0.parquet (3,596 rows)
   WROTE: redhat_parquet/redhat-inputs_WINDUPRULE_chunk0.parquet (691 rows)
   WROTE: redhat_parquet/redhat-inputs_WRKLDS_chunk0.parquet (707 rows)
   WROTE: redhat_parquet/redhat-inputs_WTO_chunk0.parquet (210 rows

In [ ]:
# ── 3b. Lectura de parquets → poblar dfs para análisis ───────────────────────
# En modo 'local' los ZIPs ya se descomprimieron y volcaron a parquet (celda anterior).
# Este bloque los lee de vuelta para poblar `dfs` y que las celdas 4-8 ejecuten.
#
# HALLAZGO: redhat-outputs.zip NO contiene issues Jira.
#   Sus CSVs tienen columnas como ['Time', 'beta', 'alpha', 'gamma'] (datos de modelo).
#   Solo redhat-inputs.zip contiene issues reales (schema: Issue key, Issue Type,
#   Status, Project key, Created, Resolved). Son 251 proyectos, ~505K issues.

import glob, gc

parquet_files = sorted(glob.glob(f'{LOCAL_OUTPUT_DIR}/redhat-inputs_*.parquet'))
print(f"Parquets de redhat-inputs encontrados: {len(parquet_files)}")

if not parquet_files:
    print("⚠️  No hay parquets. Ejecuta primero la celda de carga (MODE='local').")
    dfs = []
else:
    total_mb = sum(os.path.getsize(f) for f in parquet_files) / (1024**2)
    print(f"Tamaño total en disco: {total_mb:.1f} MB — cargando en RAM...")

    frames = []
    for pf in parquet_files:
        try:
            df_p = pd.read_parquet(pf)
            # Extraer project key del nombre: redhat-inputs_ACM_chunk0.parquet → ACM
            fname_base = os.path.basename(pf)
            project_key = fname_base[len('redhat-inputs_'):].rsplit('_chunk', 1)[0]
            df_p['_source_file'] = project_key
            df_p['_zip_source']  = 'redhat-inputs'
            frames.append(df_p)
        except Exception as e:
            print(f"  SKIP {pf}: {e}")

    if frames:
        df_all = pd.concat(frames, ignore_index=True)
        del frames
        gc.collect()
        dfs = [df_all]
        print(f"\n✅ Dataset listo para EDA:")
        print(f"   Filas totales  : {len(df_all):,}")
        print(f"   Columnas       : {list(df_all.columns)}")
        print(f"   Proyectos únicos: {df_all['_source_file'].nunique()}")
        mem_mb = df_all.memory_usage(deep=True).sum() / (1024**2)
        print(f"   Uso de RAM     : {mem_mb:.1f} MB")
    else:
        dfs = []
        print("❌ No se pudo cargar ningún parquet.")

In [ ]:
# ── 5. Distribución de tipos de issue y estados ──────────────────────────────
for df in dfs:
    fname = df['_source_file'].iloc[0] if '_source_file' in df.columns else 'dataset'
    print(f"\n=== {fname} ===")

    # Tipos de issue — 'Issue Type' es el nombre real en redhat-inputs
    type_candidates = ['Issue Type', 'issuetype', 'issue_type', 'type', 'Type', 'IssueType']
    for tc in type_candidates:
        if tc in df.columns:
            print(f"\nTipos de issue ('{tc}'):")
            print(df[tc].value_counts().head(15).to_string())

            fig, ax = plt.subplots(figsize=(10, 5))
            df[tc].value_counts().head(15).plot(kind='barh', ax=ax, color='#9b59b6')
            ax.set_title(f'Tipos de issue — Red Hat Jira (todos los proyectos)')
            ax.invert_yaxis()
            plt.tight_layout()
            plt.savefig('redhat_jira_types.png', dpi=120)
            plt.show()
            break

    # Estados
    status_candidates = ['Status', 'status', 'state', 'resolution']
    for sc in status_candidates:
        if sc in df.columns:
            print(f"\nEstados ('{sc}'):")
            print(df[sc].value_counts().head(15).to_string())
            break

In [16]:
# ── 5. Distribución de tipos de issue y estados ──────────────────────────────
for df in dfs:
    fname = df['_source_file'].iloc[0]
    print(f"\n=== {fname} ===")

    # Tipos de issue
    type_candidates = ['issuetype','issue_type','type','Type','IssueType']
    for tc in type_candidates:
        if tc in df.columns:
            print(f"\nTipos de issue ('{tc}'):")
            print(df[tc].value_counts().head(15).to_string())

            fig, ax = plt.subplots(figsize=(10, 5))
            df[tc].value_counts().head(15).plot(kind='barh', ax=ax, color='#9b59b6')
            ax.set_title(f'Tipos de issue — Red Hat Jira\n({fname})')
            ax.invert_yaxis()
            plt.tight_layout()
            plt.savefig(f'redhat_jira_types_{fname[:10]}.png', dpi=120)
            plt.show()
            break

    # Estados
    status_candidates = ['status','Status','state','resolution']
    for sc in status_candidates:
        if sc in df.columns:
            print(f"\nEstados ('{sc}'):")
            print(df[sc].value_counts().head(15).to_string())
            break

In [10]:
# ── 6. Análisis de fechas ─────────────────────────────────────────────────────
DATE_CANDIDATES = ['created','updated','resolved','resolutiondate','duedate',
                   'Created','Updated','Resolved','createdDate','updatedDate']

for df in dfs:
    fname = df['_source_file'].iloc[0]
    print(f"\n=== Fechas: {fname} ===")
    date_cols = [c for c in df.columns if c.lower() in [d.lower() for d in DATE_CANDIDATES]]
    print(f"Columnas de fecha detectadas: {date_cols}")

    for dc in date_cols[:2]:
        try:
            dates = pd.to_datetime(df[dc], dayfirst=True, errors='coerce').dropna()
            print(f"  '{dc}': {dates.min()} → {dates.max()} ({len(dates):,} válidas)")
        except Exception as e:
            print(f"  '{dc}': Error — {e}")

In [ ]:
# ── 8. Ciclos de Cycle Time ───────────────────────────────────────────────────
# IMPORTANTE: dayfirst=True obligatorio — Red Hat usa formato DD/MM/YYYY HH:MM
for df in dfs:
    fname = df['_source_file'].iloc[0] if '_source_file' in df.columns else 'dataset'
    DATE_PAIRS = [('Created', 'Resolved'), ('created', 'resolved'),
                  ('createdDate', 'resolutionDate'), ('created', 'resolutiondate')]

    for start_col, end_col in DATE_PAIRS:
        if start_col in df.columns and end_col in df.columns:
            try:
                t_start = pd.to_datetime(df[start_col], dayfirst=True, errors='coerce')
                t_end   = pd.to_datetime(df[end_col],   dayfirst=True, errors='coerce')
                cycle_time = (t_end - t_start).dt.days
                valid = cycle_time.dropna()
                valid = valid[valid >= 0]

                print(f"\nCycle Time ({start_col} → {end_col}) — todos los proyectos:")
                print(f"  Issues con resolución: {len(valid):,} / {len(df):,} ({len(valid)/len(df)*100:.1f}%)")
                print(f"  p50: {valid.median():.1f} días")
                print(f"  p75: {valid.quantile(0.75):.1f} días")
                print(f"  p90: {valid.quantile(0.90):.1f} días")
                print(f"  media: {valid.mean():.1f} días | max: {valid.max():.0f} días")

                fig, ax = plt.subplots(figsize=(10, 4))
                valid.clip(upper=valid.quantile(0.95)).hist(bins=60, ax=ax,
                                                            color='#1abc9c', edgecolor='black')
                ax.set_title('Cycle Time — Red Hat Jira (todos los proyectos)')
                ax.set_xlabel('Días'); ax.set_ylabel('Issues')
                plt.tight_layout()
                plt.savefig('redhat_jira_cycle_time.png', dpi=120)
                plt.show()
                break
            except Exception as e:
                print(f"  Error calculando cycle time: {e}")

In [12]:
# ── 8. Ciclos de Cycle Time ───────────────────────────────────────────────────
for df in dfs:
    fname = df['_source_file'].iloc[0]
    DATE_PAIRS = [('created','resolved'), ('Created','Resolved'),
                  ('createdDate','resolutionDate'), ('created','resolutiondate')]

    for start_col, end_col in DATE_PAIRS:
        if start_col in df.columns and end_col in df.columns:
            try:
                t_start = pd.to_datetime(df[start_col], errors='coerce')
                t_end   = pd.to_datetime(df[end_col], errors='coerce')
                cycle_time = (t_end - t_start).dt.days
                valid = cycle_time.dropna()
                valid = valid[valid >= 0]

                print(f"\n{fname} — Cycle Time ({start_col}→{end_col}):")
                print(f"  Issues con resolución: {len(valid):,} / {len(df):,}")
                print(f"  p50: {valid.median():.1f} días")
                print(f"  p75: {valid.quantile(0.75):.1f} días")
                print(f"  p90: {valid.quantile(0.90):.1f} días")

                fig, ax = plt.subplots(figsize=(10, 4))
                valid.clip(upper=valid.quantile(0.95)).hist(bins=50, ax=ax, color='#1abc9c', edgecolor='black')
                ax.set_title(f'Cycle Time — Red Hat Jira ({fname})')
                ax.set_xlabel('Días'); ax.set_ylabel('Issues')
                plt.tight_layout()
                plt.savefig(f'redhat_jira_cycle_time.png', dpi=120)
                plt.show()
                break
            except Exception as e:
                print(f"  Error calculando cycle time: {e}")

In [13]:
# ── 9. Resumen EDA ────────────────────────────────────────────────────────────
print("=" * 65)
print("RESUMEN EDA — RED HAT JIRA")
print("=" * 65)
print(f"\n📁 ZIPs en Bronze: {len(zip_blobs)}")
print(f"📊 DataFrames cargados: {len(dfs)}")

for df in dfs:
    fname = df['_source_file'].iloc[0]
    null_pct = (df.isnull().sum() / len(df) * 100)
    good_cols = null_pct[null_pct <= 20]
    print(f"\n   📄 {fname}")
    print(f"      Filas: {len(df):,} | Cols: {len(df.columns)}")
    print(f"      Cols completas (≤20% nulos): {len(good_cols)}/{len(df.columns)}")

print(f"""
🔧 PARÁMETROS PARA JOB SILVER RED HAT JIRA:
   - Descomprimir ZIPs en job Spark (sc.binaryFiles o extraer antes)
   - Limpiar nulos: eliminar filas sin id/key, imputar fechas faltantes
   - Estandarizar tipos de fechas a timestamps UTC
   - Calcular Cycle Time: resolved - created (en días)
   - Alinear schema con Apache Jira para tabla unificada
   - Particionar por: año de creación

🔧 PARÁMETROS PARA GOLD:
   - Mismas métricas que Apache Jira (Bug-to-Story ratio, Cycle Time)
   - Identificar proyectos Red Hat activos (prefijo key)
""")
print("✅ EDA Red Hat Jira completado")

RESUMEN EDA — RED HAT JIRA

📁 ZIPs en Bronze: 2
📊 DataFrames cargados: 0

🔧 PARÁMETROS PARA JOB SILVER RED HAT JIRA:
   - Descomprimir ZIPs en job Spark (sc.binaryFiles o extraer antes)
   - Limpiar nulos: eliminar filas sin id/key, imputar fechas faltantes
   - Estandarizar tipos de fechas a timestamps UTC
   - Calcular Cycle Time: resolved - created (en días)
   - Alinear schema con Apache Jira para tabla unificada
   - Particionar por: año de creación

🔧 PARÁMETROS PARA GOLD:
   - Mismas métricas que Apache Jira (Bug-to-Story ratio, Cycle Time)
   - Identificar proyectos Red Hat activos (prefijo key)

✅ EDA Red Hat Jira completado
